# Compute Loop Current SST Index

Downloads NOAA OISST weekly mean SST, computes the area-weighted SST anomaly over the
Caribbean/Loop Current region (lat 21–28°N, lon 270–281°E), and appends `loop_curr_sst`
as a new column to each reef-site CSV file.

**No existing data is overwritten** — only the `loop_curr_sst` column is added/updated.

**Alignment:** The CSV `year` and `week` columns follow ISO week numbering. The join
uses ISO year/ISO week from the NOAA observation dates, so year-boundary weeks
(e.g., 1988-01-03 → ISO year=1987, week=53) land in the correct CSV row.

**Re-running:** Re-run this notebook whenever new data files are added to `Data/`
or when the NOAA dataset is updated.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import xarray as xr
import pandas as pd
import fsspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

print(f'python  {sys.version.split()[0]}')
print(f'numpy   {np.__version__}')
print(f'xarray  {xr.__version__}')
print(f'pandas  {pd.__version__}')

## Configuration

In [ ]:
# ── NOAA OISST weekly SST ─────────────────────────────────────────────────────
# OPeNDAP URL: xarray opens metadata lazily, then we slice before loading.
# Only the small regional subset is downloaded (not the full global file).
NOAA_OPENDAP = 'https://psl.noaa.gov/thredds/dodsC/Datasets/noaa.oisst.v2.highres/sst.week.mean.nc'

# Fallback: direct HTTP download via fsspec (loads full file — avoid if possible)
NOAA_HTTP = 'https://downloads.psl.noaa.gov/Datasets/noaa.oisst.v2.highres/sst.week.mean.nc'

# ── Spatial bounds of the Loop Current / Caribbean index region ───────────────
# NOAA OISST uses 0–360°E longitude convention.
# 270°E = 90°W (western Gulf of Mexico), 281°E = 79°W (eastern Caribbean)
LAT_MIN, LAT_MAX = 21, 28
LON_MIN, LON_MAX = 270, 281

# ── Climatology reference period (WMO standard 1991–2020) ─────────────────────
CLIMO_START = '1991-01-01'
CLIMO_END   = '2020-12-31'

# ── Data window for the index ─────────────────────────────────────────────────
# Extend INDEX_START / leave INDEX_END as None when data files are updated.
INDEX_START = '1985-01-01'

# ── CSV files to update ───────────────────────────────────────────────────────
DATA_DIR   = Path('../Data')
SITE_FILES = {
    'sombrero': DATA_DIR / 'sombrero_preds.csv',
    'sk':       DATA_DIR / 'sk_preds.csv',
    'molasses': DATA_DIR / 'molasses_preds.csv',
}

print('Configuration loaded.')
print(f'Region: lat [{LAT_MIN}, {LAT_MAX}°N], lon [{LON_MIN}, {LON_MAX}°E]  '
      f'({360-LON_MAX}°W – {360-LON_MIN}°W)')
print(f'Climatology: {CLIMO_START} – {CLIMO_END}')

## Download NOAA OISST Data (Regional Subset Only)

Opens the dataset lazily via OPeNDAP — only metadata is fetched initially.  
The spatial slice is applied **before** `.load()`, so only the small Loop Current  
region is transferred over the network (~MB, not ~GB).

In [ ]:
print(f'Connecting via OPeNDAP: {NOAA_OPENDAP}')
try:
    ds_lazy = xr.open_dataset(NOAA_OPENDAP, engine='netcdf4')
    print('OPeNDAP connection successful.')
except Exception as e:
    print(f'OPeNDAP failed ({e}), falling back to HTTP download...')
    print('Warning: HTTP fallback downloads the full global file (~400 MB).')
    ds_lazy = xr.open_dataset(fsspec.open(NOAA_HTTP).open())

print(f'Dataset time range: {str(ds_lazy.time.values[0])[:10]} to {str(ds_lazy.time.values[-1])[:10]}')
print(f'Total time steps:   {ds_lazy.dims["time"]}')
print(f'Full lon range:     {float(ds_lazy.lon.min()):.1f}°E – {float(ds_lazy.lon.max()):.1f}°E')

# ── Slice region BEFORE loading — only this subset is downloaded ──────────────
sst_region = ds_lazy.sst.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))
print(f'\nRegion slice (lazy): {sst_region.shape}  (time × lat × lon)')
print(f'Lat: {float(sst_region.lat.min()):.2f}°N – {float(sst_region.lat.max()):.2f}°N')
print(f'Lon: {float(sst_region.lon.min()):.2f}°E – {float(sst_region.lon.max()):.2f}°E  '
      f'({360-float(sst_region.lon.max()):.1f}°W – {360-float(sst_region.lon.min()):.1f}°W)')

print('\nLoading regional data into memory...')
sst_region.load()
print(f'Done. Memory used: ~{sst_region.nbytes / 1e6:.1f} MB')

# ── Sanity check: data should not be all zeros ────────────────────────────────
mean_val = float(sst_region.mean())
std_val  = float(sst_region.std())
print(f'Data check — mean={mean_val:.2f}°C, std={std_val:.2f}°C')
if abs(mean_val) < 0.01 and std_val < 0.01:
    raise RuntimeError(
        'Data loaded as all-zeros — OPeNDAP transfer error. Re-run this cell to retry.'
    )

## Map of the Study Region

Visualize the Loop Current / Caribbean region for the first time step.

In [ ]:
# NOAA uses 0–360°E; shift to -180/180 only for display
lon_w_min = LON_MIN - 360  # -90°W
lon_w_max = LON_MAX - 360  # -79°W
sst_display = sst_region.assign_coords(lon=(sst_region.lon - 360))

fig = plt.figure(figsize=(9, 5))
ax  = plt.axes(projection=ccrs.PlateCarree())
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.LAND, facecolor='lightgray')
ax.add_feature(cfeature.BORDERS, linewidth=0.4)

sst_display.isel(time=0).plot(
    ax=ax, transform=ccrs.PlateCarree(),
    vmin=20, vmax=30, cmap='RdYlBu_r',
    cbar_kwargs={'label': 'SST (°C)', 'shrink': 0.7},
)

t0_str = str(sst_region.time.values[0])[:10]
ax.set_title(f'NOAA OISST — Loop Current / Caribbean Region — {t0_str}', fontsize=13)
ax.set_extent([lon_w_min - 5, lon_w_max + 5, LAT_MIN - 3, LAT_MAX + 3],
              crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels=True, linewidth=0.4, color='gray', alpha=0.5)
gl.top_labels = gl.right_labels = False
plt.tight_layout()
plt.show()

## Climatology and Anomalies

Computes 1991–2020 weekly climatology, then anomalies for the full record.  
**The climatology step can take several minutes — it processes the full spatial region.**

In [ ]:
# ── ISO week coordinate ────────────────────────────────────────────────────────
# xarray groupby needs a coordinate; use ISO week of year
iso_week = sst_region['time'].dt.isocalendar().week.rename('iso_week')
sst_region = sst_region.assign_coords(iso_week=iso_week)

# ── Climatology (1991–2020) ────────────────────────────────────────────────────
print('Computing 1991–2020 climatology...')
climo_period = sst_region.sel(time=slice(CLIMO_START, CLIMO_END))
climatology  = climo_period.groupby('iso_week').mean('time')
print(f'Climatology shape: {climatology.shape}  (iso_week × lat × lon)')

# Plot climatology at a central grid point
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(climatology[:, 10, 10].values, marker='o', ms=3, color='steelblue')
ax.set_xlabel('ISO week', fontsize=12)
ax.set_ylabel('SST (°C)', fontsize=12)
ax.set_title('1991–2020 Weekly Climatology (central grid point)', fontsize=12)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# ── Anomalies ──────────────────────────────────────────────────────────────────
print('Computing anomalies...')
anomalies = sst_region.groupby('iso_week') - climatology
print(f'Anomalies shape: {anomalies.shape}  (time × lat × lon)')

## Area-Weighted SST Anomaly Index

Computes the cosine-latitude-weighted spatial mean over the Loop Current region.  
This is the `loop_curr_sst` index used as a predictor in the stress prediction model.

In [ ]:
weights      = np.cos(np.deg2rad(anomalies['lat']))
index_full   = anomalies.weighted(weights).mean(('lon', 'lat'))

# Slice to the relevant period
index = index_full.sel(time=slice(INDEX_START, None))

print(f'Index time range: {str(index.time.values[0])[:10]} to {str(index.time.values[-1])[:10]}')
print(f'Index length:     {len(index)} weekly observations')

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(index.time.values, index.values, linewidth=0.8, color='steelblue')
ax.axhline(0, color='k', linewidth=0.6, linestyle='--')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('SST Anomaly (°C)', fontsize=12)
ax.set_title('Loop Current SST Anomaly Index (area-weighted mean)', fontsize=12)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## Alignment with CSV Files

**Strategy:** Join NOAA observations to CSV rows using **ISO year / ISO week**.

The CSV `year` and `week` columns follow the ISO week calendar. NOAA observations
that fall at year boundaries must be assigned by ISO year, not calendar year:

- 1988-01-03 → ISO year=1987, ISO week=53 → maps to CSV row `(1987, 53)` ✓
- Years without ISO week 53 → CSV `week=53` gets NaN (no NOAA obs for that slot)
- Summer weeks (21–40) always have valid data regardless of year type

This correctly reproduces the original `loop_curr_sst_full.csv` values.

In [ ]:
# ── Build ISO year/week lookup ────────────────────────────────────────────────
# The CSV year/week columns follow ISO year/week numbering. For example,
# 1988-01-03 is ISO year=1987, week=53 — it maps to CSV row (1987, 53),
# not (1988, 1). Using ISO year/week avoids all calendar-year drift.
df_noaa = pd.DataFrame({
    'noaa_date':     pd.DatetimeIndex(index.time.values),
    'loop_curr_sst': index.values.astype(float),
})
iso_cal = df_noaa['noaa_date'].dt.isocalendar()
df_noaa['iso_year'] = iso_cal.year.values.astype(int)
df_noaa['iso_week'] = iso_cal.week.values.astype(int)

obs_per_iso_year = df_noaa.groupby('iso_year')['iso_week'].max()
print('NOAA observations per ISO year (should be 52 or 53):')
print(obs_per_iso_year.to_string())

# ── Verify alignment against a reference CSV ──────────────────────────────────
csv_ref = pd.read_csv(SITE_FILES['sombrero'])
check = csv_ref[['year', 'week', 'totalWeek']].merge(
    df_noaa[['iso_year', 'iso_week', 'noaa_date', 'loop_curr_sst']],
    left_on=['year', 'week'],
    right_on=['iso_year', 'iso_week'],
    how='left'
).drop(columns=['iso_year', 'iso_week'])

print(f'\nCSV rows: {len(csv_ref)}  |  NOAA rows matched: {check["loop_curr_sst"].notna().sum()}')
print(f'NaN rows (no ISO week 53 that year, or future): {check["loop_curr_sst"].isna().sum()}')

print('\n--- First 5 aligned rows ---')
print(check[['year', 'week', 'totalWeek', 'noaa_date', 'loop_curr_sst']].head(5).to_string(index=False))

print('\n--- Week 52/53 boundary 1985 (no ISO week 53) ---')
m = (check['year'] == 1985) & (check['week'].isin([51, 52, 53]))
print(check.loc[m, ['year', 'week', 'totalWeek', 'noaa_date', 'loop_curr_sst']].to_string(index=False))

print('\n--- Week 52/53 boundary 1987 (HAS ISO week 53 → 1988-01-03) ---')
m = (check['year'] == 1987) & (check['week'].isin([51, 52, 53]))
print(check.loc[m, ['year', 'week', 'totalWeek', 'noaa_date', 'loop_curr_sst']].to_string(index=False))

print('\n--- Summer weeks 2024 (should be May–Oct 2024) ---')
m = (check['year'] == 2024) & (check['week'].isin([21, 25, 30, 35, 40]))
print(check.loc[m, ['year', 'week', 'totalWeek', 'noaa_date', 'loop_curr_sst']].to_string(index=False))

print('\n--- Last 5 CSV rows ---')
print(check[['year', 'week', 'totalWeek', 'noaa_date', 'loop_curr_sst']].tail(5).to_string(index=False))

## Date Alignment Verification

For rows that matched, check that the NOAA observation date falls within the expected
calendar week — i.e., within ±3 days of the expected week boundary for that year.

In [ ]:
check['noaa_date'] = pd.to_datetime(check['noaa_date'])
has_data = check[check['loop_curr_sst'].notna()].copy()

# Verify NOAA cal year matches CSV year for all matched rows
year_mismatch = has_data[has_data['noaa_date'].dt.year != has_data['year']]
print(f'Total matched rows: {len(has_data)}')
print(f'Rows where NOAA calendar year != CSV year: {len(year_mismatch)}')
if len(year_mismatch) > 0:
    # Show only week-53 boundary cases — expected for years where obs #53 is Jan 1 of next year
    print('  (these are all week=53 rows where the 53rd obs falls in Jan of next year — expected)')
    print(year_mismatch[['year', 'week', 'noaa_date']].head(10).to_string(index=False))

# Verify NOAA date is in the correct month/quarter for each CSV week
# Week 21 should be roughly May, week 40 should be roughly October
summer = has_data[(has_data['week'] >= 21) & (has_data['week'] <= 40)].copy()
summer['noaa_month'] = summer['noaa_date'].dt.month
summer_by_week = summer.groupby('week')['noaa_month'].agg(['mean', 'min', 'max'])
print('\nNOAA month for summer weeks (week 21–40):')
print(summer_by_week.to_string())

## Update CSV Files

Adds (or replaces) the `loop_curr_sst` column in each site CSV.  
All other columns are left unchanged. The column is inserted after all existing columns.

In [ ]:
for site, csv_path in SITE_FILES.items():
    df = pd.read_csv(csv_path)
    n_orig = len(df)

    # Drop existing column if present (recomputing)
    if 'loop_curr_sst' in df.columns:
        df = df.drop(columns=['loop_curr_sst'])
        print(f'{site}: replacing existing loop_curr_sst column')

    # Join on ISO year/week: CSV (year, week) matches NOAA (iso_year, iso_week).
    # This correctly handles year boundaries — e.g., 1988-01-03 is ISO year=1987
    # week=53, so it maps to CSV row (1987, 53), not (1988, 1).
    df = df.merge(
        df_noaa[['iso_year', 'iso_week', 'loop_curr_sst']],
        left_on=['year', 'week'],
        right_on=['iso_year', 'iso_week'],
        how='left',
    ).drop(columns=['iso_year', 'iso_week'])

    assert len(df) == n_orig, f'Row count mismatch for {site}! ({len(df)} vs {n_orig})'

    n_filled = df['loop_curr_sst'].notna().sum()
    n_nan    = df['loop_curr_sst'].isna().sum()
    df.to_csv(csv_path, index=False)
    print(f'{site}: {n_filled} rows filled, {n_nan} NaN (no ISO week 53 that year, or future) → {csv_path.name}')

print('\nAll CSV files updated.')

## Final Verification

Reload each CSV and confirm the new column is correct.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

for ax, (site, csv_path) in zip(axes, SITE_FILES.items()):
    df = pd.read_csv(csv_path)
    has_lcs = 'loop_curr_sst' in df.columns
    n_nan   = df['loop_curr_sst'].isna().sum() if has_lcs else 'N/A'
    print(f'{site}: loop_curr_sst present={has_lcs}, NaN={n_nan}/{len(df)}, '
          f'range=[{df["loop_curr_sst"].min():.3f}, {df["loop_curr_sst"].max():.3f}]')

    # Plot using totalWeek as x-axis for clarity
    df_plot = df.dropna(subset=['loop_curr_sst'])
    ax.plot(df_plot['totalWeek'], df_plot['loop_curr_sst'],
            linewidth=0.7, color='steelblue')
    ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
    ax.set_ylabel('loop_curr_sst', fontsize=11)
    ax.set_title(site.capitalize(), fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.4)

axes[-1].set_xlabel('totalWeek (sequential)', fontsize=11)
plt.suptitle('Loop Current SST Index Added to CSV Files', fontsize=14)
plt.tight_layout()
plt.show()

# Check alignment at a few specific weeks
print('\n--- Spot-check: sombrero week-53 boundary rows ---')
df_s = pd.read_csv(SITE_FILES['sombrero'])
check_rows = df_s[((df_s['year'] == 1985) & (df_s['week'].isin([51, 52, 53]))) |
                  ((df_s['year'] == 1986) & (df_s['week'].isin([1, 2])))]
print(check_rows[['year', 'week', 'totalWeek', 'loop_curr_sst']].to_string(index=False))